# Transformer from Scratch — Kaggle training (T4 x2)

Ready to upload to Kaggle. Settings before running: **Accelerator = GPU T4 x2**, **Internet = On**. Session max 12 hours, quota ~30 GPU-hours/week. The notebook runs in the background even if the browser closes.

Steps: install deps → clone / upload the project code → download TinyStories (200k-story subset) → train BPE tokenizer → train the 6-layer model → save checkpoint + logs → download them before the session ends.

In [ ]:
!pip install -q torch tokenizers datasets pyyaml numpy
import torch
print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), '| count:', torch.cuda.device_count())

## 1. Get the project code

Option A: clone the GitHub repo (replace the URL with the real repo URL once pushed). Option B: upload the project folder as a Kaggle dataset / attach it. Uncomment the option you use.

In [ ]:
# Option A — clone from GitHub (edit the URL)
# !git clone https://github.com/tanishq-2004/transformer-from-scratch.git
# %cd transformer-from-scratch

# Option B — if you uploaded the code as a Kaggle dataset, it appears under /kaggle/input/
import os, glob
print(glob.glob('/kaggle/input/*'))
print(os.getcwd())
print(os.listdir('.'))

## 2. Train the model (full run: 6 layers, 6 heads, 384 hidden, context 256, 200k TinyStories)

This uses `configs/small.yaml` (batch 64, lr 3e-4, warmup 500, cosine decay, grad clip 1.0, checkpoint every 1000 steps). It trains the BPE tokenizer first if `checkpoints/tokenizer.json` is missing, then encodes the data and trains. Progress, train/val loss and perplexity are printed and written to `logs/train_log.csv` / `logs/train_log.json`.

In [ ]:
!python -m src.train --config configs/small.yaml

## 3. Check the log and evaluate

Perplexity = exp(val loss), from the actual run. Generation samples use the fixed prompts in `src/evaluate.py`.

In [ ]:
!cat logs/train_log.csv
!ls -lh checkpoints/

In [ ]:
!python -m src.evaluate --checkpoint checkpoints/ckpt_step_8000.pt --tokenizer checkpoints/tokenizer.json --config configs/small.yaml --output logs/eval_final.json

## 4. Quick generation sample

In [ ]:
!python -m src.generate --checkpoint checkpoints/ckpt_step_8000.pt --tokenizer checkpoints/tokenizer.json --prompt 'Once upon a time' --max_new_tokens 50 --temperature 0.8 --top_k 50

## 5. Download before the session ends

Kaggle saves only `/kaggle/working` (20 GB). Copy the checkpoint, tokenizer and logs there, then download them from the Kaggle UI (Output tab) — the session is wiped when it ends.

If training is interrupted, resume with: `!python -m src.train --config configs/small.yaml --resume checkpoints/ckpt_step_N.pt`

In [ ]:
import shutil, os
os.makedirs('/kaggle/working/output', exist_ok=True)
for f in ['checkpoints/tokenizer.json'] + [f'checkpoints/ckpt_step_{s}.pt' for s in (8000,)]:
    if os.path.exists(f):
        shutil.copy(f, '/kaggle/working/output/')
        print('copied', f)
for f in ['logs/train_log.csv', 'logs/train_log.json', 'logs/eval_final.json']:
    if os.path.exists(f):
        shutil.copy(f, '/kaggle/working/output/')
        print('copied', f)
print(os.listdir('/kaggle/working/output'))

## Ablations (optional, after the baseline works)

Run each on a 50k-story subset, ~3k steps, changing one field at a time. Copy `configs/small.yaml`, edit `n_head` / `pos_type` / `n_layer` / `block_size` and `max_train_stories: 50000`, `max_steps: 3000`, then `!python -m src.train --config configs/ablation_XXX.yaml`. Record train/val loss and perplexity in `experiments.md`. Do NOT run full 200k training for every ablation.